In [ ]:
import numpy as np
import os
import pickle

from dotenv import load_dotenv
from sqlalchemy import create_engine
from sklearn.metrics import average_precision_score, confusion_matrix, classification_report, roc_auc_score

from customer_analytics.churn_prediction.database import ClassifierDataIterator

In [2]:
load_dotenv()
engine = create_engine(os.getenv("DATABASE_URL"))
seed = 123

In [3]:
with open("../data/classifier_techniques.pkl", "rb") as f:
    classifier_techniques = pickle.load(f)

In [ ]:
for classifier in classifier_techniques:

    best_model = classifier_techniques[classifier][
        "cv_results"
    ].best_estimator_

    optimal_threshold = classifier_techniques[classifier][
        "validation_results"
    ]["optimal_classification_threshold"]

    y_test_true = []
    y_test_pred_proba = []

    test_iterator = ClassifierDataIterator(
        engine,
        split="test",
        batch_size=10_000
    )

    for X, y in test_iterator.get_batches():

        if classifier == "support_vector_machine":
            y_pred_proba = best_model.decision_function(X)
        else:
            y_pred_proba = best_model.predict_proba(X)[:, 1]

        y_test_true.extend(y)
        y_test_pred_proba.extend(y_pred_proba)

    y_test_true = np.array(y_test_true)
    y_test_pred_proba = np.array(y_test_pred_proba)

    test_auc = roc_auc_score(
        y_test_true,
        y_test_pred_proba
    )

    test_pr_auc = average_precision_score(
        y_test_true,
        y_test_pred_proba
    )

    y_test_pred = (
        y_test_pred_proba >= optimal_threshold
    ).astype(int)

    test_classification_report = classification_report(
        y_test_true,
        y_test_pred,
        output_dict=True
    )

    test_confusion_matrix = confusion_matrix(
        y_test_true,
        y_test_pred
    )

    classifier_techniques[classifier]["test_results"] = {
        "auc": test_auc,
        "pr_auc": test_pr_auc,
        "classification_report": test_classification_report,
        "confusion_matrix": test_confusion_matrix,
        "optimal_classification_threshold": optimal_threshold,
        "optimal_f1": classifier_techniques[classifier][
            "validation_results"
        ]["optimal_f1"]
    }


In [5]:
with open("../data/classifier_techniques.pkl", "wb") as file:
    pickle.dump(classifier_techniques, file)